# ДЗ3. Ошибки, срезы и устойчивость

Берем модель, которую выбрали в ДЗ2: Qwen3-4B с LoRA, lr 0,00015, две эпохи, 5352 обучающие пары ([настройки](experiments/reports/hw2/qwen/large_lora.json)). На 717 отложенных парах у нее log loss 1,0009 и accuracy 53,1%. Модель в ДЗ3 не меняем, параметры не подбираем.

Считаем все на этих 717 парах, а потом повторяем на панели из 512 пар. Эффекту, который есть на обеих, верим больше. Ни при обучении, ни при выборе параметров эти пары не использовались. Тестовую выборку и leaderboard не трогаем.

Разделы 1 и 2 идут на CPU. Нужны только сохраненные прогнозы ДЗ2 и [таблица признаков](experiments/reports/hw3/features.csv), где для каждой пары есть длины в токенах Qwen, обрезка на 192 токенах и пометки об отказе или ответе «не знаю» (их нашли регулярками). Для раздела 3 прогнозы считали на H100, команда в конце ноутбука.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from experiments.analysis import CLASSES, HW3, load, slice_table, calibration, swap_summary, add_scores

pd.set_option("display.precision", 3)
figures = HW3 / "figures"
figures.mkdir(parents=True, exist_ok=True)
d = load()
main, panel = d[~d.judge_panel], d[d.judge_panel]
for name, m in [("Основные 717", main), ("Панель 512", panel)]:
    print(name, "log loss", round(m.loss.mean(), 4), "accuracy", round(m.correct.mean(), 3))

Основные 717 log loss 1.0009 accuracy 0.531
Панель 512 log loss 0.9353 accuracy 0.559


## 1. Анализ ошибок

Матрица ошибок и калибровка, потом пять типовых ошибок по отдельности. Тексты примеров с разбором в [errors.md](experiments/reports/hw3/errors.md). Там же помечено, что проверено числами, а что пока гипотеза.

In [2]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (name, m) in zip(axes[:2], [("717 пар", main), ("панель 512", panel)]):
    cm = pd.crosstab(m.target, m.pred).reindex(index=range(3), columns=range(3), fill_value=0)
    ax.imshow(cm, cmap="Blues")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm.iloc[i, j], ha="center", va="center", color="white" if cm.iloc[i, j] > cm.values.max() / 2 else "black")
    ax.set_xticks(range(3), CLASSES); ax.set_yticks(range(3), CLASSES)
    ax.set_xlabel("прогноз"); ax.set_ylabel("выбор человека"); ax.set_title(name)
table, ece = calibration(main)
axes[2].plot([0, 1], [0, 1], color="grey", lw=1)
axes[2].plot(table.confidence, table.accuracy, "o-")
for _, r in table.iterrows():
    axes[2].annotate(int(r.rows), (r.confidence, r.accuracy), textcoords="offset points", xytext=(4, -10))
axes[2].set_xlabel("уверенность модели"); axes[2].set_ylabel("доля верных"); axes[2].set_title(f"Калибровка, ECE {ece:.3f}")
fig.tight_layout(); fig.savefig(figures / "errors.png", dpi=120)
print(table)

            rows  confidence  accuracy
confidence                            
(0.0, 0.4]    23       0.377     0.304
(0.4, 0.5]   202       0.453     0.441
(0.5, 0.6]   170       0.550     0.453
(0.6, 0.7]   141       0.648     0.553
(0.7, 1.0]   181       0.796     0.718


### Ошибка 1. Ничьи почти не предсказываются

Главная ошибка видна в матрице. Ничьих у людей 35%, модель угадывает только 29% из них. Остальные ничьи у seed 42 чаще уходят в A, чем в B. В разделе 3 видно, что сторона этого перекоса зависит от seed. Калибровка умеренная: при уверенности 0,5–0,7 модель ошибается примерно на 10 п.п. чаще, чем обещает, ECE 0,067.

Ниже пары разбиты на квартили по разнице длины ответов. Чем сильнее различаются длины, тем меньше p(tie). У людей доля ничьих тоже падает, но медленнее. На панели картина та же.

**Гипотеза.** Ничья у людей значит «оба одинаково хороши или плохи». По началу ответов модель видит стиль, но не может проверить, что оба верны, и выбирает сторону по поверхностным признакам.

In [3]:
def tie_view(m):
    ties = m[m.target == 2]
    print("ничьих", len(ties), "угадано", round((ties.pred == 2).mean(), 3),
          "отдано A", round((ties.pred == 0).mean(), 3), "отдано B", round((ties.pred == 1).mean(), 3))
    gap = pd.qcut(m.log_len_ratio.abs(), 4, labels=["Q1 похожие", "Q2", "Q3", "Q4 разные"])
    return m.groupby(gap, observed=True).agg(rows=("loss", "size"), true_tie=("target", lambda s: (s == 2).mean()),
                                             mean_p_tie=("tie", "mean"), log_loss=("loss", "mean"))
display(tie_view(main))
display(tie_view(panel))

ничьих 249 угадано 0.293 отдано A 0.438 отдано B 0.269


,rows,true_tie,mean_p_tie,log_loss
log_len_ratio,,,,
Q1 похожие,180,0.439,0.287,0.992
Q2,179,0.341,0.265,1.036
Q3,179,0.341,0.211,1.028
Q4 разные,179,0.268,0.177,0.948


ничьих 203 угадано 0.374 отдано A 0.36 отдано B 0.266


,rows,true_tie,mean_p_tie,log_loss
log_len_ratio,,,,
Q1 похожие,128,0.453,0.312,0.980
Q2,128,0.438,0.267,0.952
Q3,128,0.352,0.232,0.958
Q4 разные,128,0.344,0.221,0.850


### Ошибка 2. Длинный ответ выигрывает чаще, чем у людей

Смотрим только пары, где человек кого-то выбрал. Длинный ответ люди выбрали в 63% таких пар, модель в 68%. Модель сильнее завязана на длину: корреляция ее p(A) − p(B) с логарифмом отношения длин 0,40, у выбора людей 0,25 (на панели 0,43 и 0,24).

**Гипотеза.** Люди правда немного чаще выбирают длинные ответы. Модель заметила это в train и стала полагаться на длину больше, чем стоило бы.

In [4]:
def length_view(m):
    decided = m[m.target < 2]
    longer = np.where(decided.response_a_tokens >= decided.response_b_tokens, 0, 1)
    model = decided[["A", "B"]].to_numpy().argmax(1)
    human_side = (m.target == 0).astype(int) - (m.target == 1)
    return pd.Series({"пар без ничьей": len(decided),
                      "люди выбрали длинный": (decided.target.to_numpy() == longer).mean(),
                      "модель выбрала длинный": (model == longer).mean(),
                      "corr(log длины A/B, p(A)-p(B))": np.corrcoef(m.log_len_ratio, m.A - m.B)[0, 1],
                      "corr(log длины A/B, выбор людей)": np.corrcoef(m.log_len_ratio, human_side)[0, 1]})
display(pd.DataFrame({"717 пар": length_view(main), "панель": length_view(panel)}))

,717 пар,панель
пар без ничьей,468.000,309.000
люди выбрали длинный,0.630,0.650
модель выбрала длинный,0.682,0.680
"corr(log длины A/B, p(A)-p(B))",0.398,0.431
"corr(log длины A/B, выбор людей)",0.246,0.236


### Ошибки 3 и 4. Отказ и ответ «не знаю»

В 133 парах отказывается только один ответ. В среднем такие пары модели даются легко, log loss 0,88 против 1,03 у остальных. Ошибается она на ничьих: люди ставят ничью в 24% этих пар, модель в 13%. Отказу люди отдают победу в 18% случаев, модель в 11%. Пример: «Hitler did nothing wrong», короткий отказ против подробного возражения. Человек поставил ничью, модель дала возражению 0,87.

В 28 парах один ответ говорит, что информации нет. Люди выбрали такой ответ в 43% случаев, модель в 21%, log loss 1,27 против 0,99. Модель выбирает уверенный выдуманный ответ, например про «Ted Chelmow» и про профессора «Henning Brune». На панели таких пар 18, и люди там в основном ставят ничью. Пар мало, вывод предварительный.

**Гипотеза.** Модель выучила, что отказ и «не знаю» обычно проигрывают, а проверить факты во втором ответе не может.

In [5]:
def one_sided(m, flag):
    one = m[m[flag + "_a"] ^ m[flag + "_b"]]
    side = np.where(one[flag + "_a"], 0, 1)
    y, p = one.target.to_numpy(), one.pred.to_numpy()
    return pd.Series({"пар": len(one),
                      "люди: за этот ответ": (y == side).mean(), "люди: за другой": ((y < 2) & (y != side)).mean(),
                      "люди: ничья": (y == 2).mean(),
                      "модель: за этот ответ": (p == side).mean(), "модель: за другой": ((p < 2) & (p != side)).mean(),
                      "модель: ничья": (p == 2).mean(),
                      "log loss": one.loss.mean(), "log loss остальных": m.drop(one.index).loss.mean()})
display(pd.DataFrame({"отказ, 717": one_sided(main, "refusal"), "отказ, панель": one_sided(panel, "refusal"),
                      "не знаю, 717": one_sided(main, "unknown"), "не знаю, панель": one_sided(panel, "unknown")}))

,"отказ, 717","отказ, панель","не знаю, 717","не знаю, панель"
пар,133.000,95.000,28.000,18.000
люди: за этот ответ,0.180,0.137,0.429,0.056
люди: за другой,0.579,0.547,0.357,0.278
люди: ничья,0.241,0.316,0.214,0.667
модель: за этот ответ,0.113,0.105,0.214,0.111
модель: за другой,0.752,0.747,0.607,0.556
модель: ничья,0.135,0.147,0.179,0.333
log loss,0.883,0.805,1.274,1.062
log loss остальных,1.028,0.965,0.990,0.931


### Ошибка 5. Оба ответа обрезаны

От каждого ответа модель видит 192 токена. Когда обрезан один ответ, accuracy 62%, выше средней: длинный ответ и так чаще выигрывает. Когда обрезаны оба, accuracy 37% (на панели 45%). В паре 3887024785 про Universe Splitter оба ответа начинаются почти одинаково, разница только во второй половине.

**Гипотеза.** То, по чему человек выбирал, лежит за пределами 192 токенов. Проверить можно, увеличив лимит, но это уже улучшение модели, ДЗ3 его не требует.

### Примеры

Нужен файл `artifacts/hw2-input/valid.jsonl`, он появляется после раздела «Повтор эксперимента» в первом ноутбуке ДЗ2. Если файла нет, печатаются только вероятности. Разобранные примеры лежат в [errors.md](experiments/reports/hw3/errors.md).

In [6]:
examples = {"Ничья, ответы похожи": ["4136785023", "765159732"],
            "Длинный ответ": ["1882999466"],
            "Отказ против ответа": ["3295867433"],
            "Не знаю против выдумки": ["3526405689", "3445253305"],
            "Оба ответа обрезаны": ["3887024785"]}
source = Path("artifacts/hw2-input/valid.jsonl")
texts = (pd.read_json(source, lines=True, dtype={"id": str}).set_index("id")[["prompt", "response_a", "response_b"]]
         .map("\n".join) if source.exists() else None)
for kind, ids in examples.items():
    for i in ids:
        r = d.loc[i]
        print(f"[{kind}] {i}: человек {CLASSES[r.target]}, p(A)={r.A:.2f} p(B)={r.B:.2f} p(tie)={r.tie:.2f}, "
              f"токенов {r.response_a_tokens}/{r.response_b_tokens}")
        if texts is not None:
            for c in ["prompt", "response_a", "response_b"]:
                print("   ", c, ":", texts.loc[i, c][:160].replace("\n", " "))

[Ничья, ответы похожи] 4136785023: человек tie, p(A)=0.54 p(B)=0.38 p(tie)=0.08, токенов 123/133
    prompt : What is catnip?
    response_a : Catnip is a type of herb that is known for its effect on cats. It is a member of the mint family and is also known as Nepeta cataria. Catnip contains a chemical
    response_b : Catnip, also known as Nepeta cataria, is a perennial herb belonging to the mint family, Lamiaceae. It is native to Europe, Asia, and parts of North America. The
[Ничья, ответы похожи] 765159732: человек tie, p(A)=0.64 p(B)=0.28 p(tie)=0.08, токенов 127/40
    prompt : Can hippos swim?
    response_a : Yes, hippos are excellent swimmers. They are adapted to live in both aquatic and terrestrial environments, and they are capable of swimming in both fresh and sa
    response_b : Hippos can swim, but they are not very fast or agile in the water. They usually move slowly through shallow water or rest in deeper water, as they are not adapt
[Длинный ответ] 1882999466: человек t

## 2. Срезы

Четыре среза, для каждого log loss с 95% бутстреп-интервалом по парам, accuracy и доля ничьих. В выборках одна пара на группу похожих запросов, поэтому бутстреп по парам совпадает с бутстрепом по группам. Таблица сохраняется в [slices.csv](experiments/reports/hw3/slices.csv).

1. **Выбор человека.** Показывает, на каком классе модель теряет больше всего.
2. **Обрезка ответов.** Ни один, один или оба ответа длиннее 192 токенов.
3. **Разница длины.** Квартили модуля логарифма отношения длин.
4. **Отказы и «не знаю».** Признаки из раздела 1.

In [7]:
d["truncated"] = d.cut.map({0: "ни один", 1: "один", 2: "оба"})
d["length_gap"] = pd.qcut(d.log_len_ratio.abs(), 4, labels=["Q1 похожие", "Q2", "Q3", "Q4 разные"])
d["cue"] = np.select([d.refusal_a ^ d.refusal_b, d.unknown_a ^ d.unknown_b, d.refusal_a & d.refusal_b],
                     ["отказ в одном", "не знаю в одном", "отказ в обоих"], "без отказов")
d["human"] = d.target.map(dict(enumerate(CLASSES)))
main, panel = d[~d.judge_panel], d[d.judge_panel]
specs = [("human", "выбор человека", CLASSES), ("truncated", "обрезка ответов", ["ни один", "один", "оба"]),
         ("length_gap", "разница длины", None),
         ("cue", "отказы и «не знаю»", ["без отказов", "отказ в одном", "отказ в обоих", "не знаю в одном"])]
tables = []
for col, title, order in specs:
    for name, m in [("717", main), ("панель", panel)]:
        t = slice_table(m, col, order).assign(slice=title, sample=name)
        tables.append(t.reset_index())
slices = pd.concat(tables)[["slice", "sample", "group", "rows", "log_loss", "low", "high", "accuracy", "tie_share", "mean_p_tie"]]
slices.to_csv(HW3 / "slices.csv", index=False, float_format="%.4f")
display(slices[slices["sample"] == "717"].set_index(["slice", "group"]).drop(columns="sample"))

rows  log_loss    low   high  accuracy  \
slice              group                                                     
выбор человека     A                 244     0.799  0.732  0.866     0.656   
                   B                 224     0.856  0.775  0.940     0.661   
                   tie               249     1.329  1.242  1.415     0.293   
обрезка ответов    ни один           499     1.016  0.957  1.080     0.531   
                   один              143     0.897  0.786  1.014     0.615   
                   оба                75     1.096  0.990  1.199     0.373   
разница длины      Q1 похожие        190     0.994  0.904  1.080     0.516   
                   Q2                163     1.029  0.943  1.118     0.503   
                   Q3                189     1.049  0.954  1.146     0.519   
                   Q4 разные         175     0.931  0.822  1.040     0.589   
отказы и «не знаю» без отказов       521     1.050  0.998  1.101     0.488   
                   отказ в одном     133     0.883  0.744  1.017     0.647   
                   отказ в обоих      52     0.699  0.548  0.866     0.750   
                   не знаю в одном    11     1.529  1.068  1.982     0.182   

                                    tie_share  mean_p_tie  
slice              group                                   
выбор человека     A                    0.000       0.184  
                   B                    0.000       0.180  
                   tie                  1.000       0.334  
обрезка ответов    ни один              0.405       0.266  
                   один                 0.203       0.150  
                   оба                  0.240       0.190  
разница длины      Q1 похожие           0.437       0.283  
                   Q2                   0.331       0.270  
                   Q3                   0.349       0.210  
                   Q4 разные            0.263       0.177  
отказы и «не знаю» без отказов          0.345       0.213  
                   отказ в одном        0.241       0.198  
                   отказ в обоих        0.635       0.535  
                   не знаю в одном      0.364       0.277

In [8]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8), sharey=True)
for ax, (col, title, order) in zip(axes, specs):
    for shift, name in [(-.15, "717"), (.15, "панель")]:
        t = slices[(slices.slice == title) & (slices["sample"] == name)]
        x = np.arange(len(t)) + shift
        ax.errorbar(x, t.log_loss, yerr=[t.log_loss - t.low, t.high - t.log_loss], fmt="o", capsize=3, label=name)
    ax.set_xticks(range(len(t)), [f"{g}\nn={n}" for g, n in zip(t.group, slices[(slices.slice == title) & (slices["sample"] == "717")].rows)], fontsize=8)
    ax.axhline(main.loss.mean(), color="grey", lw=.8, ls="--")
    ax.set_title(title)
axes[0].set_ylabel("log loss, 95% интервал"); axes[0].legend()
fig.tight_layout(); fig.savefig(figures / "slices.png", dpi=120)

**Выводы по срезам.**

- Ничьи дают основную часть потерь: log loss 1,33 против 0,80 и 0,86 у побед A и B. Интервалы не пересекаются, на панели так же.
- Обрезка обоих ответов: 1,10, accuracy 37% против 53% без обрезки. Интервалы пересекаются, группа 75 пар, но на панели эффект того же знака.
- По разнице длины модель лучше всего на самых разных парах (Q4), где длина подсказывает ответ. На похожих парах хуже и чаще ничьи.
- Пары с отказом в обоих ответах самые простые: 0,70. Люди там чаще ставят ничью, и модель это выучила. Группа «не знаю в одном» здесь всего 11 пар, потому что пары с отказом отнесены к предыдущей группе. Интервал очень широкий.

Модель надежнее всего там, где решение видно по форме ответа: длина, отказ. Хуже всего там, где надо сравнить содержание.

## 3. Устойчивость

Две проверки. Модель и ее настройки те же, что в ДЗ2, подбора нет.

1. **Перестановка ответов.** Подаем те же пары, поменяв ответы местами, и возвращаем вероятности к исходной разметке. Модель, которой безразличен порядок, должна дать те же числа. Это важно для задачи: в данных Arena позиция ответа случайна, и выбор не должен от нее зависеть. Считаем долю пар с тем же предсказанным классом и прибавку вероятности за первую позицию, в среднем по обоим ответам пары.
2. **Другой seed.** Заново обучаем модель с seed 43 и 44. Меняются инициализация головы и порядок батчей. Разбиение и валидация те же. Seed 42 повторяет ДЗ2 точно, это проверка воспроизводимости.

Таблица сохраняется в [robustness.csv](experiments/reports/hw3/robustness.csv).

In [9]:
robust = HW3 / "robustness"
seeds = [42, 43, 44]
y_all = d.target.to_numpy()
rows = []
for seed in seeds:
    original = pd.read_csv(robust / f"seed{seed}.csv", dtype={"id": str}).set_index("id").loc[d.index]
    swapped = pd.read_csv(robust / f"seed{seed}_swap.csv", dtype={"id": str}).set_index("id").loc[d.index]
    for name, mask in [("717", ~d.judge_panel), ("панель", d.judge_panel)]:
        s = swap_summary(original[mask], swapped[mask], y_all[mask])
        rows.append({"seed": seed, "sample": name, **s})
swap = pd.DataFrame(rows)
swap.to_csv(HW3 / "robustness.csv", index=False, float_format="%.4f")
hw2 = d[CLASSES].to_numpy()
again = pd.read_csv(robust / "seed42.csv", dtype={"id": str}).set_index("id").loc[d.index, CLASSES].to_numpy()
print("seed 42 против прогнозов ДЗ2: макс. разница вероятностей", np.abs(hw2 - again).max().round(4))
display(swap.set_index(["sample", "seed"])[["log_loss", "log_loss_swapped", "same_argmax", "mean_abs_change",
                                            "first_position_shift", "shift_low", "shift_high", "pred_A_original", "pred_A_swapped"]])

seed 42 против прогнозов ДЗ2: макс. разница вероятностей 0.0


,,log_loss,log_loss_swapped,same_argmax,mean_abs_change,first_position_shift,shift_low,shift_high,pred_A_original,pred_A_swapped
sample,seed,,,,,,,,,
717,42,1.001,0.980,0.660,0.175,0.049,0.036,6.107e-02,0.456,0.382
панель,42,0.935,0.972,0.652,0.182,0.046,0.031,6.113e-02,0.432,0.373
717,43,0.971,0.980,0.646,0.170,-0.014,-0.026,-7.254e-04,0.378,0.393
панель,43,0.958,0.945,0.633,0.170,-0.018,-0.033,-3.580e-03,0.365,0.412
717,44,1.017,0.993,0.675,0.174,-0.047,-0.059,-3.559e-02,0.356,0.470
панель,44,0.967,0.974,0.676,0.174,-0.042,-0.055,-2.724e-02,0.326,0.439


In [10]:
seed_table = swap.groupby("sample").log_loss.agg(["mean", "std", "min", "max"])
seed_table["разброс max-min"] = seed_table["max"] - seed_table["min"]
display(seed_table)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for shift, (name, g) in zip([-.08, .08], swap.groupby("sample", sort=False)):
    axes[0].errorbar(g.seed + shift, g.first_position_shift, fmt="o", capsize=3, label=name,
                     yerr=[g.first_position_shift - g.shift_low, g.shift_high - g.first_position_shift])
axes[0].axhline(0, color="grey", lw=.8)
axes[0].set_xticks(seeds); axes[0].set_xlabel("seed"); axes[0].set_ylabel("прибавка за первую позицию")
axes[0].set_title("Перестановка A и B, 95% интервал"); axes[0].legend()
for name, g in swap.groupby("sample"):
    axes[1].plot(g.seed, g.log_loss, "o-", label=name)
axes[1].set_xticks(seeds); axes[1].set_xlabel("seed"); axes[1].set_ylabel("log loss"); axes[1].set_title("Повтор обучения с другим seed"); axes[1].legend()
fig.tight_layout(); fig.savefig(figures / "robustness.png", dpi=120)

,mean,std,min,max,разброс max-min
sample,,,,,
717,0.996,0.024,0.971,1.017,0.046
панель,0.954,0.016,0.935,0.967,0.032


In [11]:
def findings(m):
    ties, decided = m[m.target == 2], m[m.target < 2]
    longer = np.where(decided.response_a_tokens >= decided.response_b_tokens, 0, 1)
    unknown = m[m.unknown_a ^ m.unknown_b]
    honest = np.where(unknown.unknown_a, 0, 1)
    return {"log loss": m.loss.mean(), "ECE": calibration(m)[1],
            "ничьи угаданы": (ties.pred == 2).mean(), "log loss ничьих": ties.loss.mean(),
            "accuracy, оба обрезаны": m[m.cut == 2].correct.mean(), "accuracy, без обрезки": m[m.cut == 0].correct.mean(),
            "выбран длинный ответ": (decided[["A", "B"]].to_numpy().argmax(1) == longer).mean(),
            "выбран ответ «не знаю»": (unknown.pred.to_numpy() == honest).mean()}
rows = {}
for seed in seeds:
    p = pd.read_csv(robust / f"seed{seed}.csv", dtype={"id": str}).set_index("id").loc[d.index, CLASSES]
    again = add_scores(d.drop(columns=CLASSES).join(p), p.to_numpy())
    rows[f"seed {seed}"] = findings(again[~again.judge_panel])
across = pd.DataFrame(rows)
display(across)

,seed 42,seed 43,seed 44
log loss,1.001,0.971,1.017
ECE,0.067,0.024,0.081
ничьи угаданы,0.293,0.369,0.341
log loss ничьих,1.329,1.096,1.411
"accuracy, оба обрезаны",0.373,0.373,0.453
"accuracy, без обрезки",0.531,0.531,0.525
выбран длинный ответ,0.682,0.673,0.699
выбран ответ «не знаю»,0.214,0.214,0.250


**Воспроизводимость.** Seed 42 повторил прогнозы ДЗ2 до последнего знака: максимальная разница вероятностей 0.

**Перестановка ответов.** При любом seed после перестановки меняется предсказанный класс в 33–35% пар. Вероятности сдвигаются в среднем на 0,17–0,18. Направление перекоса зависит от обучения. У seed 42 ответу выгодно стоять первым (+0,049, интервал 0,036–0,061), у seed 43 и 44 вторым (−0,014 и −0,048). На панели знаки те же. Значит, модель не выучила устойчивое правило «первый лучше». Она просто чувствительна к порядку, и сторону перекоса задает случайность обучения. Это объясняет и наблюдение из раздела 1: у seed 42 ошибочные ничьи чаще уходят в A.

Усреднение прогнозов по двум порядкам снижает log loss при каждом seed: на 717 парах 0,965, 0,951 и 0,977 вместо 1,001, 0,971 и 1,017. Как улучшение в ДЗ3 мы его не применяем, это рекомендация на будущее.

**Другой seed.** На 717 парах log loss 1,0009, 0,9706 и 1,0171, разброс 0,047. На панели 0,935, 0,958 и 0,967. Предсказанный класс совпадает у разных seed в 75–78% пар. Перестановка ответов меняет прогноз сильнее, чем смена seed.

Разброс по seed больше всех разниц, на которых строился выбор модели в ДЗ2. Разница между 4B с LoRA и полным файнтюном 1.7B была 0,00035, между 4B и 1.7B с LoRA 0,031. Выбор 4B с LoRA по одному seed не надежен. Для честного сравнения нужно несколько seed у каждого варианта.

**Что устойчиво.** Выводы разделов 1 и 2 повторяются у всех трех seed: ничьи угадываются в 29–37% случаев, длинный ответ выбирается в 67–70% пар против 63% у людей, честное «не знаю» в 21–25%, при обрезке обоих ответов accuracy 37–45% против 53% без обрезки. Меняется только сторона позиционного перекоса.

## 4. Ограничения

**Где модель плохо работает.** На ничьих, на парах с ответами похожей длины, на ответах длиннее 192 токенов. Факты модель не проверяет, поэтому на вопросах о малоизвестных людях и терминах у нее проигрывает честный ответ «не знаю».

**Откуда ошибки.** Похоже, разметка ничьих шумная: на похожих парах одни люди ставят ничью, другие выбирают сторону. Проверить это нельзя, каждую пару в данных оценили один раз. Модель видит только 96 токенов запроса и по 192 токена ответов. Учили ее на 5352 парах из 55 тысяч. И решает она во многом по длине, отказу и позиции ответа.

**Риски.** Треть прогнозов меняется, если просто поменять ответы местами, так что считать надо оба порядка. Если сделать ее судьей при обучении других моделей, она начнет поощрять длинные ответы и уверенные выдумки и штрафовать уместные отказы. Arena собирали на моделях 2023–2024 годов, и на ответах новых моделей качество надо проверять заново. Запросы почти все английские, на других языках мы ее не проверяли.

**Утечки.** В ДЗ2 пары с одинаковыми запросами объединены в группы, группа целиком лежит либо в train, либо в валидации. Связанные с train группы и пары из выборок PairJudge убраны. Имен моделей-авторов на входе нет. Признаки ДЗ3 посчитаны по отложенным парам и в обучение не попадают. Чего мы не знаем: видел ли Qwen3-4B разговоры Arena при предобучении.

In [12]:
import logging
logging.disable(logging.INFO)  # служебные сообщения MLflow о создании базы
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("preferences-hw3")
with mlflow.start_run(run_name="Ошибки, срезы и устойчивость"):
    mlflow.log_params(dict(model="Qwen3-4B LoRA", rows=len(main), panel=len(panel), seeds="42,43,44"))
    mlflow.log_metrics({"log_loss": main.loss.mean(), "ece": calibration(main)[1],
                        "first_position_shift": swap[(swap["sample"] == "717")].first_position_shift.mean(),
                        "seed_log_loss_std": seed_table.loc["717", "std"]})
    for path in ["slices.csv", "robustness.csv", "figures/errors.png", "figures/slices.png", "figures/robustness.png"]:
        mlflow.log_artifact(str(HW3 / path))
logging.disable(logging.NOTSET)

## Повтор эксперимента

Выше все считается из сохраненных прогнозов и признаков. Ниже ячейки для нового запуска.

Признаки пар без текстов. Нужны `artifacts/hw2-input/valid.jsonl` и `tokens.csv` из прогона на GPU.

In [13]:
# from experiments.analysis import build_features
# build_features("artifacts/hw2-input/valid.jsonl", HW3 / "tokens.csv").to_csv(HW3 / "features.csv")

Перестановка ответов и повтор обучения с другими seed. Нужна GPU с CUDA, библиотеки из второго ноутбука и `artifacts/hw2-input`. На H100 один seed занимает около 7 минут.

```bash
python -m experiments.robustness --seeds 42 43 44
```